# Generating a Spectral Index Data Cube with Cubo and Spyndex
## Introduction
Using the cubo and spyndex library by David Montero Loaiza (Github: [davemlz](https://github.com/davemlz)), we can easily create spectral index data cubes from coordinates and visualize them.

In [2]:
!pip install spyndex cubo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.4/770.4 kB 3.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 27.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 25.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [cubo]7m━━━━━━━━━ 10/13 [pystac_client]


In [3]:
import lexcube
import numpy as np
import xarray as xr
import spyndex
import cubo

Using [cubo](https://github.com/ESDS-Leipzig/cubo), we can create a data cube from Sentinel 2 using STAC:

In [17]:
bands_da = cubo.create(
    lat=12.43, # Central latitude of the cube
    lon=-2.29, # Central longitude of the cube
    collection="sentinel-2-l2a", # Name of the STAC collection
    bands=["B01","B02","B03","B04","B05","B06","B07","B08","B11","B12"], # Bands to retrieve
    start_date="2022-01-25", # Start date of the cube
    end_date="2022-12-31", # End date of the cube
    edge_size=1024, # Edge size of the cube (px)
    resolution=10, # Pixel size of the cube (m)
    query={"eo:cloud_cover": {"lt": 10} }
)

Now, we can use that data cube to compute any spectral index (in this case NDVI) using the [spyndex](https://github.com/awesome-spectral-indices/spyndex) library:

In [18]:
da = spyndex.computeIndex(
    index = ["NDVI"],
    params = {
        "N": bands_da.sel(band = "B08"),
        "R": bands_da.sel(band = "B04"),
        "L": 0.5
    }
)

Finally, we can visualize the NDVI cube with Lexcube:

In [11]:
# w = lexcube.Cube3DWidget(da)
# w.plot()

Afterwards, it is possible to parametrize the visualization further or save the figure as a PNG:

In [20]:
w.overlay_geojson("test/cube_example/sah_admin_0ocha.geosjon")

Opening GeoJSON from file...
Loaded GeoJSON from file test/cube_example/sah_admin_0ocha.geosjon.


In [9]:
##Save the figure as PNG:
w.savefig()
##Or adjust the colormap:
w.vmin = -0.1
w.vmax = 0.5
w.cmap = "thermal"

When using Lexcube and generated images or videos, please acknowledge/cite: Söchting, M., Scheuermann, G., Montero, D., & Mahecha, M. D. (2025). Interactive Earth system data cube visualization in Jupyter notebooks. Big Earth Data, 1–15. https://doi.org/10.1080/20964471.2025.2471646


In [10]:
w.save_print_template()

When using Lexcube and generated images or videos, please acknowledge/cite: Söchting, M., Scheuermann, G., Montero, D., & Mahecha, M. D. (2025). Interactive Earth system data cube visualization in Jupyter notebooks. Big Earth Data, 1–15. https://doi.org/10.1080/20964471.2025.2471646


## With GEE for Temp

In [39]:
# !pip install ee geemap
# !pip install --upgrade xee

In [1]:
import lexcube
import xarray as xr
import ee

In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize(project="ee-kizarzakir",
                 opt_url='https://earthengine-highvolume.googleapis.com')


In [5]:
ds = xr.open_dataset('ee://ECMWF/ERA5_LAND/HOURLY', engine='ee',
                         crs='EPSG:4326', scale=0.25, chunks={})

In [6]:
da = ds["temperature_2m"][630000:630003,2:1438,2:718]
da

<xarray.DataArray 'temperature_2m' (time: 3, lon: 1436, lat: 716)> Size: 12MB
dask.array<getitem, shape=(3, 1436, 716), dtype=float32, chunksize=(3, 256, 256), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 24B 2021-11-14T01:00:00 ... 2021-11-14T03:...
  * lon      (lon) float64 11kB -179.4 -179.1 -178.9 ... 178.9 179.1 179.4
  * lat      (lat) float64 6kB -89.38 -89.12 -88.88 -88.62 ... 88.88 89.12 89.38
Attributes:
    id:             temperature_2m
    data_type:      {'type': 'PixelType', 'precision': 'double'}
    dimensions:     [3601, 1801]
    crs:            EPSG:4326
    crs_transform:  [0.1, 0, -180.05, 0, -0.1, 90.05]

In [12]:
w = lexcube.Cube3DWidget(da)
w.plot()

Cube3DWidget(api_metadata={'/api': {'status': 'ok', 'api_version': 5}, '/api/datasets': [{'id': 'default', 'sh…

## Example with much larger time frame
You can alternatively load and visualize a much larger time frame (seven years instead of one), which may take 2-3 minutes to load:

In [ ]:
from sen2nbar.nbar import nbar_cubo

In [ ]:
lbands_da = cubo.create(
    lat=51, # Central latitude of the cube
    lon=10, # Central longitude of the cube
    collection="sentinel-2-l2a", # Name of the STAC collection
    bands=["B01","B02","B03","B04","B05","B06","B07","B08","B11","B12"], # Bands to retrieve
    start_date="2016-01-01", # Start date of the cube
    end_date="2022-12-31", # End date of the cube
    edge_size=1024, # Edge size of the cube (px)
    resolution=10, # Pixel size of the cube (m)
    query={"eo:cloud_cover": {"lt": 10} }
)
lbands_da = nbar_cubo(lbands_da)

lda = spyndex.computeIndex(
    index = ["NDVI"],
    params = {
        "N": lbands_da.sel(band = "B08"),
        "R": lbands_da.sel(band = "B04"),
        "L": 0.5
    }
)

In [ ]:
lw = lexcube.Cube3DWidget(lda)
lw

In [ ]:
# ** Save the figure as PNG:
# lw.savefig()
# ** Or adjust the colormap:
# lw.vmin = -0.1
# lw.vmax = 0.5